# Module 10 - Pretraining Notebook

Use this notebook after the Module 03B training-dynamics utilities and Module 10 training utilities are implemented. The focus is the pretraining loop: multi-position batches, language-model cross-entropy, trainer wiring, and small training runs on text.

In [ ]:
from __future__ import annotations

import contextlib
import io
import math
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch

import g2c
from g2c.tokenizer import BPETokenizer
from g2c.training import Trainer, clip_grad_norm_, cosine_with_warmup, get_lm_batch, lm_cross_entropy
from g2c.transformer import TransformerLM

_ = torch.manual_seed(0)
repo_root = Path(g2c.__file__).resolve().parents[1]
trainer_device = "auto"
print("MPS available:", torch.backends.mps.is_available())


## Before the Notebook

Module 03B should already provide `AdamW`, `cosine_with_warmup`, and `clip_grad_norm_`. For Module 10, implement `lm_cross_entropy` and `Trainer.train_step`. The trainer also depends on your completed `TransformerLM` from Module 09.

In [ ]:
"Run from the terminal: .venv/bin/python -m pytest tests/test_training_dynamics.py -x"
"Then run: .venv/bin/python -m pytest tests/test_training.py -x"
"Question: Which training test is the next one failing, and which utility does it point at?"
"Answer: "

In [ ]:
for test_file in ["tests/test_training_dynamics.py", "tests/test_training.py"]:
    result = subprocess.run(
        [sys.executable, "-m", "pytest", test_file],
        cwd=repo_root,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    if result.stderr:
        print(result.stderr)
    assert result.returncode == 0, f"{test_file} is not passing yet."

print("Module 03B training dynamics and Module 10 training tests passed.")

## Exercise 1 - Multi-Position Batches

A transformer gets `T` next-token targets from each length-`T` window. Check the shift directly before thinking about the model.

In [ ]:
ids = torch.arange(30) % 10
xb, yb = get_lm_batch(ids, batch_size=4, context_length=6, generator=torch.Generator().manual_seed(0))

print("x shape:", tuple(xb.shape))
print("y shape:", tuple(yb.shape))
print("first x row:", xb[0].tolist())
print("first y row:", yb[0].tolist())
print("y - x for first row:", (yb[0] - xb[0]).tolist())
assert torch.equal(yb, (xb + 1) % 10)

In [ ]:
"Question: Why does one (B, T) batch provide B*T classification examples instead of B examples?"
"Answer: "
"Question: Why is the target for x[:, t] equal to the next token, not the same token?"
"Answer: "

## Exercise 2 - Language-Model Cross-Entropy

With uniform logits, the loss should be `log(vocab_size)`. Keep this number in mind when reading training curves: random-init models should start near this baseline.

In [ ]:
B, T, V = 3, 4, 7
uniform_logits = torch.zeros(B, T, V)
targets = torch.randint(0, V, (B, T), generator=torch.Generator().manual_seed(1))
loss = lm_cross_entropy(uniform_logits, targets)

print("loss:", float(loss.item()))
print("log(V):", math.log(V))
assert abs(loss.item() - math.log(V)) < 1e-5

In [ ]:
"Question: What reshape turns logits from (B, T, V) into the shape CrossEntropyLoss already expects?"
"Answer: "
"Question: What bug would you suspect if step-0 loss starts far below log(V)?"
"Answer: "

## Module 03B Review - Warmup Plus Cosine Decay

The schedule should already be implemented. Plot it once here so the trainer's learning-rate history is easy to interpret later.

In [ ]:
schedule_max_steps = 500
schedule_warmup_steps = 50
schedule_max_lr = 3e-4
schedule_min_lr = 3e-5

steps = list(range(schedule_max_steps + 1))
lrs = [
    cosine_with_warmup(
        step,
        warmup_steps=schedule_warmup_steps,
        max_steps=schedule_max_steps,
        max_lr=schedule_max_lr,
        min_lr=schedule_min_lr,
    )
    for step in steps
]

plt.figure(figsize=(8, 4))
plt.plot(steps, lrs)
plt.xlabel("step")
plt.ylabel("learning rate")
plt.title("Linear warmup + cosine decay")
plt.show()

print("step 0 lr:", lrs[0])
print("last warmup lr:", lrs[schedule_warmup_steps - 1])
print("max step lr:", lrs[-1])

In [ ]:
"Question: In Module 03B terms, what kind of curve would suggest the learning rate is too high?"
"Answer: "
"Question: Where does the scheduled learning rate get written during Trainer.train_step?"
"Answer: "

## Module 03B Review - Global Gradient Clipping

Clipping should already be implemented. This is a quick sanity check before watching `grad_norm` during a real pretraining run.

In [ ]:
p1 = torch.zeros(1, requires_grad=True)
p2 = torch.zeros(1, requires_grad=True)
p1.grad = torch.tensor([3.0])
p2.grad = torch.tensor([4.0])

before = math.sqrt(float((p1.grad ** 2).sum() + (p2.grad ** 2).sum()))
returned_norm = clip_grad_norm_([p1, p2], max_norm=1.0)
after = math.sqrt(float((p1.grad ** 2).sum() + (p2.grad ** 2).sum()))

print("before:", before)
print("returned pre-clip norm:", returned_norm)
print("after:", after)
print("p1 grad:", p1.grad.item(), "p2 grad:", p2.grad.item())

assert abs(returned_norm - 5.0) < 1e-3
assert after <= 1.0 + 1e-3

In [ ]:
"Question: Where should clipping happen relative to backward() and optimizer.step()?"
"Answer: "
"Question: If clipping fires on nearly every step, what hyperparameter would you inspect first?"
"Answer: "

## Exercise 3 - Prepare TinyShakespeare or Fallback Text

Use the downloaded TinyShakespeare file if setup has it. The fallback keeps the notebook runnable, but TinyShakespeare is the intended corpus.

In [ ]:
def load_pretraining_text(max_chars: int = 200_000) -> str:
    path = repo_root / "data" / "tinyshakespeare.txt"
    if path.exists():
        return path.read_text(encoding="utf-8")[:max_chars]

    base = """
    FIRST STUDENT:
    the model predicts the next token from the context.
    SECOND STUDENT:
    the context helps the model choose a better next token.
    FIRST STUDENT:
    gradients update embeddings and transformer blocks.
    """
    return ("\n".join(line.strip() for line in base.strip().splitlines()) + "\n") * 1000


text = load_pretraining_text()
tokenizer = BPETokenizer()
target_vocab_size = 1024
with contextlib.redirect_stdout(io.StringIO()):
    tokenizer.train(text, vocab_size=target_vocab_size)

all_ids = torch.tensor(tokenizer.encode(text), dtype=torch.long)
vocab_size = len(tokenizer.vocab)
split = int(0.9 * len(all_ids))
train_ids = all_ids[:split]
val_ids = all_ids[split:]

print("characters:", len(text))
print("tokens:", len(all_ids))
print("vocab size:", vocab_size)
print("uniform-loss baseline:", math.log(vocab_size))
print("train tokens:", len(train_ids))
print("val tokens:", len(val_ids))
assert len(val_ids) > 128

In [ ]:
"Question: Why can a larger BPE vocabulary reduce token count but make the output logits tensor wider?"
"Answer: "
"Question: Why should validation text come from held-out token positions rather than the same windows used for training?"
"Answer: "

## Exercise 4 - Train a Tiny TransformerLM

This is the first real pretraining loop. Start with the small defaults, verify loss drops, then increase `max_steps`, `embedding_dim`, `num_layers`, or corpus size when you want a longer run.

In [ ]:
torch.manual_seed(0)
model = TransformerLM(
    vocab_size=vocab_size,
    embedding_dim=64,
    num_layers=2,
    num_heads=4,
    max_seq_len=64,
    hidden_dim=128,
)

trainer = Trainer(
    model,
    batch_size=16,
    context_length=48,
    max_steps=400,
    max_lr=3e-4,
    min_lr=3e-5,
    warmup_steps=40,
    weight_decay=0.01,
    grad_clip=1.0,
    eval_every=50,
    eval_iters=5,
    log_every=10,
    device=trainer_device,
    optimizer="adamw",
    generator=torch.Generator().manual_seed(0),
)

print("training device:", trainer.device)
history = trainer.train(train_ids, val_ids)
print("final train loss:", history["train_loss"][-1])
print("final val loss:", history["val_loss"][-1])
print("final val perplexity:", math.exp(history["val_loss"][-1]))

In [ ]:
def plot_training_history(history: dict[str, list], *, baseline_loss: float | None = None) -> None:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(history["step"], history["train_loss"], label="train")
    if history["val_loss"]:
        axes[0].plot(history["val_step"], history["val_loss"], marker="o", label="val")
    if baseline_loss is not None:
        axes[0].axhline(baseline_loss, color="gray", linestyle="--", label="log(V)")
    axes[0].set_xlabel("step")
    axes[0].set_ylabel("cross entropy")
    axes[0].legend()

    axes[1].plot(history["step"], history["lr"])
    axes[1].set_xlabel("step")
    axes[1].set_ylabel("learning rate")

    axes[2].plot(history["step"], history["grad_norm"])
    axes[2].set_xlabel("step")
    axes[2].set_ylabel("pre-clip grad norm")

    fig.tight_layout()
    plt.show()


plot_training_history(history, baseline_loss=math.log(vocab_size))

In [ ]:
"Question: Does validation loss track training loss, or is one moving much faster?"
"Answer: "
"Question: Are gradient norms largest early in training? What does clipping seem to be doing?"
"Answer: "

## Exercise 5 - Sample During or After Training

Module 11 will build the real generation utilities. For now, use a minimal local sampler so you can qualitatively inspect what pretraining learned.

Byte-level BPE can sample raw byte tokens before the model is well trained. The raw escaped sample shows that behavior directly; the readable sample uses top-k sampling and blocks control/invalid-byte tokens so the output is easier to inspect.

In [ ]:
def token_is_readable(tokenizer: BPETokenizer, token_id: int) -> bool:
    piece = tokenizer.decode([token_id])
    if not piece or "�" in piece:
        return False
    return all(ch in "\n\t" or ch.isprintable() for ch in piece)


def readable_token_mask(tokenizer: BPETokenizer) -> torch.Tensor:
    return torch.tensor(
        [token_is_readable(tokenizer, token_id) for token_id in range(len(tokenizer.vocab))],
        dtype=torch.bool,
    )


def apply_top_k(logits: torch.Tensor, top_k: int | None) -> torch.Tensor:
    if top_k is None:
        return logits
    k = min(top_k, logits.numel())
    cutoff = torch.topk(logits, k).values[-1]
    return logits.masked_fill(logits < cutoff, float("-inf"))


@torch.no_grad()
def sample_text(
    model: TransformerLM,
    tokenizer: BPETokenizer,
    prompt: str,
    *,
    max_new_tokens: int = 200,
    temperature: float = 0.7,
    top_k: int | None = 50,
    printable_only: bool = True,
    seed: int = 0,
) -> str:
    device = next(iter(model.parameters())).device
    ids = torch.tensor(tokenizer.encode(prompt), dtype=torch.long, device=device)
    generator = torch.Generator().manual_seed(seed)
    printable_mask = readable_token_mask(tokenizer) if printable_only else None

    for _ in range(max_new_tokens):
        ctx = ids[-model.max_seq_len :].unsqueeze(0)
        logits = model(ctx)[0, -1].clone()

        if printable_mask is not None:
            mask = printable_mask.to(device=logits.device)
            logits = logits.masked_fill(~mask, float("-inf"))

        if temperature == 0.0:
            next_id = logits.argmax().reshape(1)
        else:
            logits = apply_top_k(logits / temperature, top_k)
            probs = torch.softmax(logits, dim=-1).detach().cpu()
            next_id = torch.multinomial(probs, 1, generator=generator).to(device)
        ids = torch.cat([ids, next_id.to(device=ids.device)])

    return tokenizer.decode(ids.detach().cpu().tolist())


def escaped_text(text: str) -> str:
    return text.encode("unicode_escape", errors="backslashreplace").decode("ascii")


raw_sample = sample_text(
    model,
    tokenizer,
    "KING:",
    max_new_tokens=300,
    temperature=0.8,
    top_k=None,
    printable_only=False,
    seed=1,
)
readable_sample = sample_text(
    model,
    tokenizer,
    "KING:",
    max_new_tokens=300,
    temperature=0.7,
    top_k=50,
    printable_only=True,
    seed=1,
)

print("raw sample, escaped so control bytes do not wreck the notebook output")
print("-" * 72)
print(escaped_text(raw_sample))
print("\nreadable top-k sample")
print("-" * 72)
print(readable_sample)

In [ ]:
"Question: How different is the raw escaped sample from the readable filtered sample?"
"Answer: "

"Question: What improved first: punctuation and line shape, local words, or global meaning?"
"Answer: "

"Question: If the readable sample is still incoherent, is that more likely a sampling issue or a training-budget/model-size issue?"
"Answer: "

## Exercise 6 - Learning-Rate Sweep

This sweep uses fewer steps than the module's full exercise so it is practical inside a notebook. After the shape of the result makes sense, scale the step count up.

In [ ]:
def run_lr_sweep(max_lrs: list[float], *, steps: int = 150) -> dict[float, float]:
    results: dict[float, float] = {}
    for lr in max_lrs:
        torch.manual_seed(1)
        sweep_model = TransformerLM(
            vocab_size=vocab_size,
            embedding_dim=64,
            num_layers=2,
            num_heads=4,
            max_seq_len=64,
            hidden_dim=128,
        )
        sweep_trainer = Trainer(
            sweep_model,
            batch_size=16,
            context_length=48,
            max_steps=steps,
            max_lr=lr,
            min_lr=lr / 10,
            warmup_steps=max(1, steps // 10),
            weight_decay=0.01,
            grad_clip=1.0,
            eval_every=steps,
            eval_iters=5,
            log_every=steps,
            device=trainer_device,
            optimizer="adamw",
            generator=torch.Generator().manual_seed(123),
        )
        sweep_history = sweep_trainer.train(train_ids, val_ids)
        results[lr] = sweep_history["val_loss"][-1]
        print(f"max_lr={lr:g} val_loss={results[lr]:.3f} perplexity={math.exp(results[lr]):.1f}")
    return results


sweep_lrs = [1e-4, 3e-4, 1e-3, 3e-3]
lr_results = run_lr_sweep(sweep_lrs, steps=150)

plt.figure(figsize=(7, 4))
plt.plot(list(lr_results.keys()), list(lr_results.values()), marker="o")
plt.xscale("log")
plt.xlabel("max_lr")
plt.ylabel("final val loss")
plt.title("Learning-rate sweep")
plt.show()

In [ ]:
"Question: Which max_lr was best in this short sweep?"
"Answer: "
"Question: Which side of the U-shape is underfitting, and which side is instability?"
"Answer: "

## Exercise 7 - Warmup and Clipping Ablations

Run short matched experiments with one safety feature removed. The goal is not a definitive benchmark; it is to see the failure signature in the curves.

In [ ]:
def run_ablation(name: str, *, warmup_steps: int, grad_clip: float | None, steps: int = 150) -> dict[str, list]:
    torch.manual_seed(2)
    ablation_model = TransformerLM(
        vocab_size=vocab_size,
        embedding_dim=64,
        num_layers=2,
        num_heads=4,
        max_seq_len=64,
        hidden_dim=128,
    )
    ablation_trainer = Trainer(
        ablation_model,
        batch_size=16,
        context_length=48,
        max_steps=steps,
        max_lr=3e-4,
        min_lr=3e-5,
        warmup_steps=warmup_steps,
        weight_decay=0.01,
        grad_clip=grad_clip,
        eval_every=steps,
        eval_iters=5,
        log_every=10,
        device=trainer_device,
        optimizer="adamw",
        generator=torch.Generator().manual_seed(321),
    )
    out = ablation_trainer.train(train_ids, val_ids)
    out["name"] = name
    return out


ablations = [
    run_ablation("warmup + clip", warmup_steps=15, grad_clip=1.0),
    run_ablation("no warmup", warmup_steps=0, grad_clip=1.0),
    run_ablation("no clipping", warmup_steps=15, grad_clip=None),
]

plt.figure(figsize=(8, 4))
for item in ablations:
    plt.plot(item["step"], item["train_loss"], label=item["name"])
plt.axhline(math.log(vocab_size), color="gray", linestyle="--", label="log(V)")
plt.xlabel("step")
plt.ylabel("train loss")
plt.title("Warmup and clipping ablations")
plt.legend()
plt.show()

In [ ]:
"Question: Which ablation changed the first 50 steps the most?"
"Answer: "
"Question: Did no-clipping show larger gradient norm spikes, or was this tiny run too small to expose them?"
"Answer: "